In [1]:
"""Очистка 2026-09-Database-Fe-Ni-Co-Al-Cr-Ti.xlsx:
шапка как в августовском файле, кириллица-двойники -> латиница, лишние пробелы,
фаза Хойслера -> L21, названия методов в Preparation через дефис.
Исходные файлы не изменяются: результат пишется в новый *_cleaned.xlsx, правки - в лог.
Форматирование и формулы сохраняются (правка через openpyxl)."""
from pathlib import Path
import re
import pandas as pd
from openpyxl import load_workbook

SRC = "2026-09-Database-Fe-Ni-Co-Al-Cr-Ti.xlsx"          # единственный файл, который чистим
HEADER_SRC = "2026-08-Database-Fe-Ni-Co-Al-Cr-Ti_renumbered.xlsx"   # только читаем шапку, август не меняется
DST = Path(SRC).with_name(Path(SRC).stem + "_cleaned.xlsx")
LOG = Path(SRC).with_name(Path(SRC).stem + "_cleaning_log.xlsx")

# кириллические буквы, внешне неотличимые от латинских
HOMOGLYPHS = str.maketrans({
    "А": "A", "В": "B", "С": "C", "Е": "E", "Н": "H", "К": "K", "М": "M", "О": "O", "Р": "P",
    "Т": "T", "Х": "X", "а": "a", "с": "c", "е": "e", "о": "o", "р": "p", "х": "x", "у": "y",
    "к": "k", "і": "i"})
# если после замены остаются настоящие русские буквы, ячейку не трогаем, а пишем в лог
CYRILLIC = re.compile(r"[\u0400-\u04FF]")
HEUSLER = re.compile(r"Heusler\s*(?:\(\s*L2[_]?1\s*\))?", re.IGNORECASE)

log = []


def record(cell, col, old, new, rule):
    log.append({"Ячейка": cell.coordinate, "Строка": cell.row, "Столбец": col,
                "Было": old, "Стало": new, "Правило": rule})


def norm_preparation(v):
    """Шаги процесса разделены ' + ', внутри шага пробелы -> '-', лишние дефисы по краям убираются.
    'induction melting + casting Cu' -> 'induction-melting + casting-Cu'
    'arc-melting + -induction-melting' -> 'arc-melting + induction-melting'
    'splat quenching (induction levitation)' -> 'splat-quenching (induction-levitation)'"""
    steps = []
    for step in v.split("+"):
        step = step.strip(" -")
        if not step:
            continue
        step = re.sub(r"\s+\(", "\x00(", step)         # пробел перед скобкой сохраняем
        step = re.sub(r"\s+", "-", step).replace("\x00", " ")
        step = re.sub(r"-{2,}", "-", step)
        steps.append(step)
    return " + ".join(steps)


def clean_text(value, col, is_data_col):
    """Возвращает (новое значение, [правила])."""
    rules, v = [], value
    if is_data_col:
        t = v.translate(HOMOGLYPHS)
        if t != v:
            if CYRILLIC.search(t):                # остались настоящие русские буквы
                rules.append("ВНИМАНИЕ: смешанная кириллица, вручную")
            else:
                v, _ = t, rules.append("кириллица -> латиница")
        t = HEUSLER.sub("L21", v)
        if t != v:
            v, _ = t, rules.append("Heusler -> L21")
    t = v.replace("\xa0", " ").replace("\t", " ").replace("\n", " " if is_data_col else "\n")
    if is_data_col:
        t = re.sub(r"\s+", " ", t)
        if col == "Phase composition":
            t = re.sub(r"\s*\+\s*", " + ", t)
    else:
        t = re.sub(r"[ ]{2,}", " ", t)
    t = t.strip()
    if t != v:
        v, _ = t, rules.append("пробелы")
    if is_data_col and col == "Preparation":
        t = norm_preparation(v)
        if t != v:
            v, _ = t, rules.append("Preparation: дефисы")
    return v, rules


wb = load_workbook(SRC)                        # без data_only: формулы =SUM(...) сохраняются
ws = wb.active
aug_header = [c.value for c in load_workbook(HEADER_SRC).active[1]]
while aug_header and aug_header[-1] is None:   # отбрасываем пустые хвостовые ячейки шапки
    aug_header.pop()
n_data = len(aug_header)                       # 38 столбцов данных, дальше - заметки

# --- 1. шапка: первые 38 ячеек = августовская шапка (точно, с её пробелами) ---
for i, new in enumerate(aug_header, start=1):
    cell = ws.cell(1, i)
    if cell.value != new:
        record(cell, new, cell.value, new, "шапка как в августе")
        cell.value = new

# --- 2. остальные ячейки ---
for row in ws.iter_rows():
    for cell in row:
        v = cell.value
        if not isinstance(v, str) or v.startswith("="):
            continue
        if cell.row == 1 and cell.column <= n_data:
            continue                           # шапку уже заменили
        col = aug_header[cell.column - 1] if cell.column <= n_data else str(ws.cell(1, cell.column).value)
        col_clean = col.strip() if col else col
        new, rules = clean_text(v, col_clean, cell.column <= n_data)
        if new != v:
            record(cell, col_clean, v, new, ", ".join(rules))
            cell.value = new
        elif rules:
            record(cell, col_clean, v, v, ", ".join(rules))

wb.save(DST)
log_df = pd.DataFrame(log, columns=["Ячейка", "Строка", "Столбец", "Было", "Стало", "Правило"])
with pd.ExcelWriter(LOG, engine="openpyxl") as xw:
    log_df.to_excel(xw, sheet_name="Лог", index=False)
    (log_df.groupby("Правило").size().rename("Ячеек").reset_index()
     .to_excel(xw, sheet_name="Сводка", index=False))

print(f"Сохранено: {DST}\nЛог изменений ({len(log_df)}): {LOG}")
print(log_df.groupby("Правило").size().to_string() if len(log_df) else "изменений нет")

Сохранено: 2026-09-Database-Fe-Ni-Co-Al-Cr-Ti_cleaned.xlsx
Лог изменений (136): 2026-09-Database-Fe-Ni-Co-Al-Cr-Ti_cleaning_log.xlsx
Правило
Heusler -> L21                   8
Preparation: дефисы             47
пробелы                         33
пробелы, Preparation: дефисы    43
шапка как в августе              5
